# 06 Results
Prepares summary of the models results from notebook 05_models. No retraining is done.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

results = pd.read_csv("../outputs/results_all.csv")
print(len(results))

99


## AUROC Table

Prepares a table to show the mean ± standard deviation of the AUROC score over the folds (antibiotic x model x split). The time split has only one fold, so std is not defined.

In [2]:
auroc_summary = (
    results
    .groupby(["antibiotic", "model", "split"])["AUROC"]
    .agg(["mean", "std"])
    .round(3)
)

auroc_summary

mean    std
antibiotic model        split                
CIP        boosting     lineage  0.912  0.043
                        random   0.967  0.010
                        time     0.955    NaN
           lineage_only lineage  0.500  0.000
                        random   0.931  0.010
                        time     0.922    NaN
           logistic     lineage  0.883  0.047
                        random   0.959  0.010
                        time     0.944    NaN
CXM        boosting     lineage  0.864  0.050
                        random   0.890  0.023
                        time     0.814    NaN
           lineage_only lineage  0.500  0.000
                        random   0.793  0.030
                        time     0.753    NaN
           logistic     lineage  0.842  0.038
                        random   0.887  0.022
                        time     0.814    NaN
GEN        boosting     lineage  0.967  0.013
                        random   0.965  0.020
                        time     0.991    NaN
           lineage_only lineage  0.500  0.000
                        random   0.794  0.026
                        time     0.760    NaN
           logistic     lineage  0.947  0.009
                        random   0.958  0.012
                        time     0.964    NaN

In [3]:
def format_cell(row):
    if pd.isna(row['std']):
        return f"{row['mean']:.3f}"
    return f"{row['mean']:.3f} ± {row['std']:.3f}"

auroc_table = auroc_summary.apply(format_cell, axis=1).unstack("split")
auroc_table = auroc_table[["random", "lineage", "time"]]
auroc_table

split                           random        lineage   time
antibiotic model                                            
CIP        boosting      0.967 ± 0.010  0.912 ± 0.043  0.955
           lineage_only  0.931 ± 0.010  0.500 ± 0.000  0.922
           logistic      0.959 ± 0.010  0.883 ± 0.047  0.944
CXM        boosting      0.890 ± 0.023  0.864 ± 0.050  0.814
           lineage_only  0.793 ± 0.030  0.500 ± 0.000  0.753
           logistic      0.887 ± 0.022  0.842 ± 0.038  0.814
GEN        boosting      0.965 ± 0.020  0.967 ± 0.013  0.991
           lineage_only  0.794 ± 0.026  0.500 ± 0.000  0.760
           logistic      0.958 ± 0.012  0.947 ± 0.009  0.964

### Interpretation

The gradient boosting model has the highest AUROC for all three antibiotics, in both the random and the lineage split. In the random split, the difference to logistic regression is small (under 0.01), so the two models perform about equally. In the lineage split, boosting is higher by about 0.02–0.03, it suggests that boosting performs better on families it has not seen.

The standard deviation (std) under the random split is small for every antibiotic (0.010–0.030), with 0.022–0.023 being the std for CXM. Under the lineage split, it increases for CIP (from 0.010 to 0.043–0.047) and CXM (goes up to 0.038–0.050), but decreases for GEN (by around 0.003–0.007). This difference probably happens because, in the lineage split, each family stays in one fold, so folds end up being more different from each other. The largest family, where most strains are resistant to CIP and CXM but not GEN, is put in one fold.

Under the lineage split, the baseline scores exactly 0.500 ± 0.000 for every antibiotic. This confirms that no family information passes from training to test strains. Under the random split, the baseline comes close to the score of the gene model for CIP (0.931 against 0.959 and 0.967), but it is much lower for CXM (0.793 against 0.887 and 0.890) and GEN (0.794 against 0.958 and 0.965). So, for CIP, family information explains most of gene models' performance on a random split.

Under the time split, CIP scores are close to the random split (0.944–0.955), and the baseline alone reaches 0.922, so the same families seem to persist over time. CXM scores lowest here (0.814), possibly because its resistance changed between the periods. Since the time split has only one test set, these results should be read with caution.

## Balanced Accuracy Table

Prepares a table to show the mean ± standard deviation of the balanced accuracy score over the folds (antibiotic x model x split). The time split has only one fold, so std is not defined.

In [4]:
ba_summary = (
    results
    .groupby(["antibiotic", "model", "split"])["balanced_accuracy"]
    .agg(["mean", "std"])
    .round(3)
)

ba_table = ba_summary.apply(format_cell, axis=1).unstack("split")
ba_table = ba_table[["random", "lineage", "time"]]
ba_table

split                           random        lineage   time
antibiotic model                                            
CIP        boosting      0.901 ± 0.016  0.743 ± 0.113  0.898
           lineage_only  0.847 ± 0.022  0.500 ± 0.000  0.845
           logistic      0.910 ± 0.010  0.751 ± 0.128  0.895
CXM        boosting      0.842 ± 0.023  0.808 ± 0.049  0.732
           lineage_only  0.693 ± 0.028  0.500 ± 0.000  0.677
           logistic      0.827 ± 0.020  0.775 ± 0.029  0.740
GEN        boosting      0.922 ± 0.034  0.899 ± 0.027  0.969
           lineage_only  0.680 ± 0.032  0.500 ± 0.000  0.653
           logistic      0.893 ± 0.010  0.856 ± 0.027  0.900

### Interpretation

In both splits, the gradient boosting model performs better for CXM and GEN, while logistic scores slightly higher for CIP (by 0.009 for random and 0.008 for lineage). However, the lineage gap is tiny compared to its std (0.113 and 0.128) so the two models perform about the same under balanced accuracy. For GEN, the difference in performance is the highest (random: 0.922 vs. 0.893, lineage: 0.899 vs. 0.856).

The standard deviation under the random split lies between 0.010 and 0.034. Under the lineage split, it becomes very large for CIP (0.113–0.128), much larger than in the AUROC table (0.043–0.047). This is probably because balanced accuracy depends on the 0.5 cut-off. The fold containing the largest family has many more resistant strains than the training data, so more strains fall on the wrong side of the cut-off, even when the model still ranks them reasonably well.

Overall, balanced accuracy shows the same picture as AUROC: CIP drops the most from the random to the lineage split (from about 0.90–0.91 to about 0.75), while GEN drops the least. The baseline again scores exactly 0.500 under the lineage split. Under the time split, CXM again has its lowest scores (0.732–0.740).

## Drop between random and lineage splits

Prepares a table for the drop in AUROC score between random and lineage splits.

- drop = random score − lineage score

A drop close to 0 means the model works just as well on new families, while larger drop shows that the model's resistance prediction relies on knowing the family. 

In [5]:
means = results.groupby(["antibiotic", "model", "split"])["AUROC"].mean().unstack("split")
means["drop"] = means["random"] - means["lineage"]
means[["random", "lineage", "drop"]].round(3)

split                    random  lineage   drop
antibiotic model                               
CIP        boosting       0.967    0.912  0.056
           lineage_only   0.931    0.500  0.431
           logistic       0.959    0.883  0.076
CXM        boosting       0.890    0.864  0.026
           lineage_only   0.793    0.500  0.293
           logistic       0.887    0.842  0.044
GEN        boosting       0.965    0.967 -0.002
           lineage_only   0.794    0.500  0.294
           logistic       0.958    0.947  0.011

### Interpretation

For both models, the drop values follow the order: GEN < CXM < CIP.
<br>
Boosting drop:
-0.002 < 0.026 < 0.056
<br>
Logistic drop:
0.011 < 0.044 < 0.076

This is the order the hypothesis predicts. Logistic regression drops more than boosting for every antibiotic, which fits with boosting holding up better on new families. For GEN, the boosting drop is negative, so boosting scores slightly higher on new families and shows no drop at all.

Compared with the standard deviation under the lineage split, the CIP drop (0.056–0.076) is a bit larger than the variation between folds (0.043–0.047). The CXM drop is about the same size as its variation (0.038–0.050), so it is less certain, and the GEN drop is within the variation.

The baseline's drop is its random-split score minus 0.5, so it shows how much family alone can predict. It is largest for CIP (0.431) and almost the same for CXM and GEN (0.293 and 0.294). So family is most informative for CIP, which is also the antibiotic whose gene models drop most when families are held out.

Overall, the results support the hypothesis, but they are based on five folds without a statistical test.